In [8]:
# Step 1: Install required dependencies
%pip install -q pandas numpy matplotlib seaborn

In [9]:
import os
import sys
import glob
from pathlib import Path
import pandas as pd
import numpy as np

# Step 2: Configure Destination Directory
# In Google Colab, mount Google Drive; otherwise, use local directory
try:
    from google.colab import drive
    print("Detected Google Colab environment. Mounting Google Drive...")
    drive.mount('/content/drive')
    
    colab_candidates = [
        "/content/drive/MyDrive/capstone/mealRec",
        "/content/drive/MyDrive/capstone/MealRec",
        "/content/drive/MyDrive/capstone/mealrec",
        "/content/drive/MyDrive/mealRec"
    ]
    DEST_DIR = next((p for p in colab_candidates if os.path.exists(p)), colab_candidates[0])
except ImportError:
    print("Running in local environment.")
    local_candidates = [
        os.path.abspath("mydrive/capstone/mealRec"),
        os.path.abspath("mydrive/capstone/MealRec"),
        os.path.abspath("../mydrive/capstone/mealRec"),
        os.path.abspath("mealRec")
    ]
    DEST_DIR = next((p for p in local_candidates if os.path.exists(p)), local_candidates[0])

os.makedirs(DEST_DIR, exist_ok=True)
print(f"Target folder ready: {DEST_DIR}")
print(f"Folder exists: {os.path.exists(DEST_DIR)}")

Detected Google Colab environment. Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Target folder ready: /content/drive/MyDrive/capstone/mealRec
Folder exists: True


In [10]:
# Step 3: Scan directory and list all available files
print(f"Scanning files in {DEST_DIR}...\n" + "=" * 60)

all_files = []
for root, dirs, files in os.walk(DEST_DIR):
    for f in files:
        if not f.startswith('.'):
            full_path = os.path.join(root, f)
            rel_path = os.path.relpath(full_path, DEST_DIR)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            all_files.append((rel_path, full_path, size_mb))

if all_files:
    print(f"Found {len(all_files)} file(s):\n")
    for rel_path, full_path, size_mb in sorted(all_files):
        print(f"  • {rel_path:<40} ({size_mb:.2f} MB)")
else:
    print(f"Notice: No files currently found in '{DEST_DIR}'.")
    print("If your files are uploaded to Google Drive or local mydrive, ensure the folder name matches 'mealRec'.")

Scanning files in /content/drive/MyDrive/capstone/mealRec...
Found 4 file(s):

  • Copy of meal.csv                         (0.09 MB)
  • Copy of recipe.csv                       (673.34 MB)
  • Copy of user_meal.csv                    (0.62 MB)
  • Copy of user_recipe.csv                  (6.07 MB)


In [11]:
# Step 4: Define Smart Loader for MealRec dataset formats (.txt, .csv, .tsv, .json)
def load_mealrec_file(file_path):
    """
    Loads tabular and text files commonly used in MealRec and MealRec+.
    Automatically assigns appropriate column names and parses delimiters.
    """
    fname = os.path.basename(file_path).lower()
    
    # Handle JSON
    if fname.endswith('.json'):
        try:
            return pd.read_json(file_path)
        except Exception:
            return pd.read_json(file_path, lines=True)

    # Peek at first line for delimiter detection
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        first_line = f.readline()
        
    sep = '\t' if '\t' in first_line else (',' if ',' in first_line else r'\s+')
    
    # Check if first line contains alphabetical characters indicating existing header
    first_tokens = first_line.strip().split() if sep == r'\s+' else first_line.strip().split(sep)
    has_header = any(t.isalpha() and not t.isdigit() for t in first_tokens)
    
    # Map standard MealRec schema for unheadered files
    names = None
    if not has_header:
        if 'user_course' in fname:
            names = ['user_id', 'course_id']
        elif 'course_category' in fname:
            names = ['course_id', 'category_id'] # 0: Appetizer, 1: Main, 2: Dessert
        elif 'user_meal' in fname:
            if len(first_tokens) == 2:
                names = ['user_id', 'meal_id']
            elif len(first_tokens) == 3:
                names = ['user_id', 'meal_id', 'rating']
            elif len(first_tokens) >= 4:
                names = ['user_id', 'meal_id', 'rating', 'timestamp'][:len(first_tokens)]
        elif 'meal_course' in fname:
            # Meal-course affiliation format often: meal_id course_1 course_2 course_3
            names = [f'col_{i}' for i in range(len(first_tokens))]
            if len(names) >= 2:
                names[0] = 'meal_id'
                for i in range(1, len(names)):
                    names[i] = f'course_{i}'
        elif 'course_fsa' in fname:
            names = ['fsa_score']
        elif 'course_who' in fname:
            names = ['who_score']

    try:
        if names:
            df = pd.read_csv(file_path, sep=sep, names=names, engine='python')
        else:
            df = pd.read_csv(file_path, sep=sep, engine='python')
        return df
    except Exception as e:
        print(f"Warning: Fallback loading for {fname}: {e}")
        return pd.read_csv(file_path, sep=None, engine='python')

In [12]:
# Step 5: Ingest all discovered files into Pandas DataFrames
datasets = {}

for rel_path, full_path, size_mb in all_files:
    ext = os.path.splitext(full_path)[1].lower()
    if ext in ['.txt', '.csv', '.tsv', '.json']:
        # Clean key name
        key = os.path.splitext(rel_path)[0].replace('/', '__').replace('\\', '__')
        try:
            df = load_mealrec_file(full_path)
            datasets[key] = df
            print(f"✓ Loaded: '{key}' | Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
        except Exception as err:
            print(f"✗ Error loading '{rel_path}': {err}")

print(f"\nTotal datasets successfully loaded: {len(datasets)}")

✓ Loaded: 'Copy of meal' | Shape: 3,817 rows x 4 columns
✗ Error loading 'Copy of recipe.csv': field larger than field limit (131072)
✓ Loaded: 'Copy of user_meal' | Shape: 46,768 rows x 2 columns
✓ Loaded: 'Copy of user_recipe' | Shape: 151,148 rows x 4 columns

Total datasets successfully loaded: 3


In [13]:
# Step 6: Detailed Inspection and Previews
if not datasets:
    print("No datasets currently loaded. Once files are in the target directory, re-run this step.")
else:
    for name, df in datasets.items():
        print("=" * 70)
        print(f"Dataset: {name}")
        print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
        print(f"Columns: {list(df.columns)}")
        null_counts = {col: int(cnt) for col, cnt in df.isnull().sum().items() if cnt > 0}
        print(f"Missing values: {null_counts if null_counts else 'None'}")
        print("-" * 70)
        try:
            from IPython.display import display
            display(df.head())
        except ImportError:
            print(df.head())
        print("\n")

Dataset: Copy of meal
Shape: 3,817 rows x 4 columns
Columns: ['bundle_id', 'appetizer', 'main_dish', 'dessert']
Missing values: None
----------------------------------------------------------------------


,bundle_id,appetizer,main_dish,dessert
0,242,8626,8652,17652
1,885,15034,25272,24843
2,1077,24745,26460,17652
3,1187,15184,55872,19017
4,1340,13062,13218,10813




Dataset: Copy of user_meal
Shape: 46,768 rows x 2 columns
Columns: ['user_id', 'meal_id']
Missing values: None
----------------------------------------------------------------------


,user_id,meal_id
0,meal_id,user_id
1,26,39
2,37,39
3,218,2079
4,534,8952




Dataset: Copy of user_recipe
Shape: 151,148 rows x 4 columns
Columns: ['user_id', 'recipe_id', 'rating', 'dateLastModified']
Missing values: None
----------------------------------------------------------------------


,user_id,recipe_id,rating,dateLastModified
0,39,61727,4,2001-03-26T07:36:16.653\n
1,39,7612,5,2004-08-02T16:48:37.107\n
2,39,12009,4,2001-02-28T10:37:25.59\n
3,39,88185,5,2009-05-18T14:55:11.517\n
4,39,24445,5,2015-05-07T15:42:20.32\n


In [14]:
# Step 7: MealRec Key Entities & Exploratory Data Analysis
print("=== MealRec Recommendation System Key Statistics ===\n")

# 1. User-Meal Interactions
um_key = next((k for k in datasets if 'user_meal' in k.lower()), None)
if um_key:
    df_um = datasets[um_key]
    u_col = next((c for c in df_um.columns if 'user' in str(c).lower()), df_um.columns[0])
    m_col = next((c for c in df_um.columns if 'meal' in str(c).lower()), df_um.columns[1])
    print(f"[User-Meal Interactions ({um_key})]")
    print(f"  • Total records:         {len(df_um):,}")
    print(f"  • Unique users:          {df_um[u_col].nunique():,}")
    print(f"  • Unique meals:          {df_um[m_col].nunique():,}")
    if 'rating' in df_um.columns:
        print(f"  • Rating mean:           {df_um['rating'].mean():.2f} (std: {df_um['rating'].std():.2f})")
        print(f"  • Rating distribution:   {df_um['rating'].value_counts().to_dict()}")
    print()

# 2. User-Course Interactions
uc_key = next((k for k in datasets if 'user_course' in k.lower()), None)
if uc_key:
    df_uc = datasets[uc_key]
    u_col = next((c for c in df_uc.columns if 'user' in str(c).lower()), df_uc.columns[0])
    c_col = next((c for c in df_uc.columns if 'course' in str(c).lower()), df_uc.columns[1])
    print(f"[User-Course Interactions ({uc_key})]")
    print(f"  • Total records:         {len(df_uc):,}")
    print(f"  • Unique users:          {df_uc[u_col].nunique():,}")
    print(f"  • Unique courses:        {df_uc[c_col].nunique():,}")
    print()

# 3. Course Categories
cat_key = next((k for k in datasets if 'category' in k.lower()), None)
if cat_key:
    df_cat = datasets[cat_key]
    print(f"[Course Categories ({cat_key})]")
    cat_col = df_cat.columns[-1]
    cat_map = {0: 'Appetizer', 1: 'Main Dish', 2: 'Dessert'}
    for cat_val, count in df_cat[cat_col].value_counts().items():
        label = cat_map.get(cat_val, f'Category {cat_val}')
        print(f"  • {label}: {count:,} courses")
    print()

# 4. Meal-Course Bundle Affiliations
mc_key = next((k for k in datasets if 'meal_course' in k.lower()), None)
if mc_key:
    df_mc = datasets[mc_key]
    print(f"[Meal-Course Bundles ({mc_key})]")
    print(f"  • Total meals with courses: {len(df_mc):,}")
    print()

if not any([um_key, uc_key, cat_key, mc_key]):
    print("Standard MealRec files (user_meal, user_course, meal_course, course_category) not identified yet.")

=== MealRec Recommendation System Key Statistics ===

[User-Meal Interactions (Copy of user_meal)]
  • Total records:         46,768
  • Unique users:          3,818
  • Unique meals:          1,576

